# 🌿 AgroVision AI — EfficientNet Model Training
**Dataset:** New Plant Diseases Dataset (PlantVillage) — 87,000+ images, 38 classes

Natija: `plant_disease_model.pth` fayli → `backend/models_weights/` ga ko'chirasiz

In [ ]:
# 1. Dataset yuklab olish
import subprocess
subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
print('✅ Setup done')

In [ ]:
# 2. Dataset import (Kaggle notebook ichida)
import os
# Kaggle notebook ichida dataset quyidagi yo'lda bo'ladi:
DATASET_PATH = '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VALID_DIR = os.path.join(DATASET_PATH, 'valid')

# Sinflar sonini ko'rish
classes = sorted(os.listdir(TRAIN_DIR))
print(f'✅ Jami sinflar: {len(classes)}')
for i, c in enumerate(classes[:5]):
    imgs = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    print(f'  {i}: {c} ({imgs} rasm)')

In [ ]:
# 3. Kutubxonalar
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import json
import time
from pathlib import Path

# GPU bor-yo'qligini tekshirish
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 4. Data transforms
IMG_SIZE = 224
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VALID_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes
print(f'✅ Train: {len(train_dataset)} rasm')
print(f'✅ Valid: {len(val_dataset)} rasm')
print(f'✅ Sinflar: {NUM_CLASSES}')

In [ ]:
# 5. EfficientNet-B2 modeli (B0 dan aniqroq, B4 dan tezroq)
def create_model(num_classes: int):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    
    # Classifier qayta o'rnatamiz
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model

model = create_model(NUM_CLASSES).to(DEVICE)
print(f'✅ Model tayyor: EfficientNet-B2')
print(f'   Parametrlar: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 6. Training setup
EPOCHS = 15
LR = 1e-3

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'✅ Training: {EPOCHS} epoch, lr={LR}')

In [ ]:
# 7. Training loop
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


best_val_acc = 0.0
history = []

print('🚀 Training boshlanmoqda...\n')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    elapsed = time.time() - t0
    
    history.append({'epoch': epoch, 'train_acc': train_acc, 'val_acc': val_acc})
    
    star = ' ⭐' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_plant_disease_model.pth')
    
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train: {train_acc:.1%} ({train_loss:.3f}) | '
          f'Val: {val_acc:.1%} ({val_loss:.3f}) | '
          f'{elapsed:.0f}s{star}')

print(f'\n✅ Training tugadi! Eng yaxshi validatsiya: {best_val_acc:.1%}')

In [ ]:
# 8. Sinf nomlarini saqlash (backend uchun kerak)
class_info = {
    'classes': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'model': 'efficientnet_b2',
    'img_size': IMG_SIZE,
    'best_val_acc': best_val_acc,
}

with open('class_names.json', 'w') as f:
    json.dump(class_info, f, indent=2)

print('✅ Fayllar saqlandi:')
print('   - best_plant_disease_model.pth  ← ASOSIY MODEL')
print('   - class_names.json              ← sinf nomlari')
print()
print('📥 Output bo\'limidan ikkala faylni yuklab oling!')

In [ ]:
# 9. Model test — bitta rasm bilan sinab ko'rish
from PIL import Image
import torch.nn.functional as F

def predict_image(model, image_path, class_names, device, transform):
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs = F.softmax(outputs, dim=1)
        top5_probs, top5_indices = torch.topk(probs, 5)
    print('Top-5 natijalar:')
    for prob, idx in zip(top5_probs[0], top5_indices[0]):
        print(f'  {class_names[idx]}: {prob.item():.1%}')

# Validatsiya papkasidan birinchi rasmni olamiz
first_class = os.listdir(VALID_DIR)[0]
first_img = os.listdir(os.path.join(VALID_DIR, first_class))[0]
test_path = os.path.join(VALID_DIR, first_class, first_img)

print(f'Test rasm: {first_class}/{first_img}')

# Best model ni yuklaymiz
model.load_state_dict(torch.load('best_plant_disease_model.pth', map_location=DEVICE))
predict_image(model, test_path, CLASS_NAMES, DEVICE, val_transforms)